# Auxiliary ASR Diagonal Attention Evaluation

In [ ]:

# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        device_name = torch.cuda.get_device_name(i)
        devices.append({
            'type': 'CUDA',
            'available': True,
            'name': device_name,
            'index': i
        })
else:
    devices.append({'type': 'CUDA', 'available': False, 'name': 'N/A'})
devices


In [ ]:

# change folder into the root of the ASR project
import os

if not os.path.isdir('Configs'):
    %cd ../

!pwd


In [ ]:

# import packages, define helper utilities
import os
import yaml
import torch
import torch.nn as nn
import pandas as pd

from models import ASRCNN
from meldataset import build_dataloader
from token_map import build_token_map_from_data


def cfg_get_nested(cfg: dict, path, default=None, sep="."):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur


def prepare_model_params(config: dict, vocab_size: int) -> dict:
    defaults = {
        "input_dim": 80,
        "hidden_dim": 256,
        "n_token": vocab_size,
        "token_embedding_dim": 512,
        "n_layers": 5,
        "decoder_type": "conformer",
        "decoder_config": {},
    }

    raw_params = cfg_get_nested(config, "model_params", {}) or {}
    if not isinstance(raw_params, dict):
        raw_params = {}

    params = {
        **defaults,
        **{k: v for k, v in raw_params.items()
           if k not in {"decoder", "decoder_type", "decoder_config", "location_kernel_size"}},
    }

    params["n_token"] = int(raw_params.get("n_token", params.get("n_token", vocab_size)))
    if params["n_token"] <= 0:
        params["n_token"] = vocab_size

    decoder_type = raw_params.get("decoder_type")
    decoder_config = raw_params.get("decoder_config") or {}
    decoder_entry = raw_params.get("decoder")

    if isinstance(decoder_entry, dict):
        decoder_config = decoder_entry
        decoder_type = decoder_entry.get("type", decoder_type)

    if decoder_type is None and "location_kernel_size" in raw_params:
        decoder_type = "lstm"
    if decoder_type is None:
        decoder_type = defaults.get("decoder_type", "conformer")

    if decoder_type == "lstm":
        lstm_cfg = decoder_config.get("lstm", {})
        kernel = raw_params.get("location_kernel_size", lstm_cfg.get("location_kernel_size"))
        if kernel is not None:
            lstm_cfg = {**lstm_cfg, "location_kernel_size": kernel}
        if "random_mask_prob" in raw_params:
            lstm_cfg = {**lstm_cfg, "random_mask_prob": raw_params["random_mask_prob"]}
        decoder_config = {**decoder_config, "lstm": lstm_cfg}

    params["decoder_type"] = decoder_type if decoder_type in {"lstm", "conformer"} else "conformer"
    params["decoder_config"] = decoder_config

    return params


def load_token_map_from_config(config):
    token_src = config.get("phoneme_maps_path")
    if not token_src:
        return build_token_map_from_data(
            config.get("train_data"),
            config.get("val_data"),
            config.get("ood_data"),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {word: index for word, index in csv}


def load_asr_model(model_path, config_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    token_map = load_token_map_from_config(config)

    model_params = prepare_model_params(config, len(token_map))

    model = ASRCNN(**model_params)
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("model", checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized_state = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized_state)

    model.to(device)
    model.eval()
    return model, config, token_map


def build_dev_dataloader(config, device, batch_size=None, num_workers=None):
    val_csv_path = config.get("val_data")
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    with open(val_csv_path, "r", encoding="utf-8") as f:
        raw_lines = [line.rstrip("
") for line in f]

    path_list = []
    for raw in raw_lines:
        if not raw.strip():
            continue
        parts = raw.split("|")
        if len(parts) == 1:
            continue
        path = parts[0]
        if len(parts) == 2:
            text = parts[1]
            speaker = ""
        else:
            text = "|".join(parts[1:-1])
            speaker = parts[-1]
        path_list.append([path, text, speaker])

    if batch_size is None:
        batch_size = int(cfg_get_nested(config, "eval_params.batch_size", cfg_get_nested(config, "batch_size", 4)))
    if num_workers is None:
        num_workers = int(cfg_get_nested(config, "dataloader_params.val_num_workers", 2))

    dataset_params = {
        "dict_path": cfg_get_nested(config, "phoneme_maps_path", "Data/word_index_dict.txt"),
        "sr": cfg_get_nested(config, "preprocess_params.sr", 24000),
        "spect_params": cfg_get_nested(
            config,
            "preprocess_params.spect_params",
            {"n_fft": 1024, "win_length": 1024, "hop_length": 300},
        ),
        "mel_params": cfg_get_nested(config, "preprocess_params.mel_params", {"n_mels": 80}),
    }

    collate_config = {"return_wave": False}
    device_flag = device.type if isinstance(device, torch.device) else str(device)
    loader = build_dataloader(
        path_list=path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device_flag,
        collate_config=collate_config,
        dataset_config=dataset_params,
    )
    return loader



In [ ]:
# load model, config, and validation dataloader
checkpoint_dir = 'Checkpoint'
config_path = 'Checkpoint/config.yml'

if not os.path.isdir(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint directory '{checkpoint_dir}' not found.")

ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.startswith('epoch_') and f.endswith('.pth')]
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoint files found in '{checkpoint_dir}'.")

ckpt_files = sorted(ckpt_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
model_path = os.path.join(checkpoint_dir, ckpt_files[-1])

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print(f'model --> {model_path}')
print(f'config --> {config_path}')
phoneme_source = config.get('phoneme_maps_path', 'built from dataset')
print(f'dictionary --> {phoneme_source}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, token_map = load_asr_model_from_config(config, model_path, device)

dev_loader, val_entries = build_dev_dataloader_from_config(config, device)
print(f'Validation dataset size: {len(val_entries)} samples')
print(f'Batch size --> {dev_loader.batch_size}')


In [ ]:
# attention alignment diagnostics
import numpy as np

@torch.no_grad()
def diagonal_attention_score(model: ASRCNN, dev_loader, device: Optional[torch.device] = None, band: float = 0.1, max_batches: Optional[int] = None):
    """Compute the average diagonal attention concentration score.

    Args:
        model: ASR model returning alignment matrices when teacher-forced.
        dev_loader: validation dataloader yielding (texts, text_lens, mels, mel_lens).
        device: device for computation; defaults to model parameters' device.
        band: allowable deviation from the diagonal (0-1 range).
        max_batches: limit the number of batches to evaluate (useful for quick checks).

    Returns:
        mean_score: average ratio of attention mass inside the diagonal band.
        scores: list of per-utterance scores.
    """
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    diag_scores: List[float] = []
    downsample = 2 ** getattr(model, 'n_down', 1)

    for batch_idx, batch in enumerate(dev_loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        texts, text_lens, mels, mel_lens = batch[:4]
        mels = mels.to(device)
        texts = texts.to(device)
        text_lens = text_lens.to(torch.long)
        mel_lens = mel_lens.to(device=device, dtype=torch.long)

        reduced_mel_lens = torch.clamp(mel_lens // downsample, min=1)
        mel_mask = model.length_to_mask(reduced_mel_lens)

        outputs = model(mels, src_key_padding_mask=mel_mask, text_input=texts)
        if isinstance(outputs, dict):
            attn = outputs.get('attn') or outputs.get('attention') or outputs.get('alignments')
            if attn is None:
                raise KeyError('Model output dictionary does not contain attention matrices.')
        elif isinstance(outputs, (tuple, list)):
            if len(outputs) < 3:
                raise ValueError('Model forward output does not include attention tensors.')
            attn = outputs[2]
        else:
            raise TypeError('Unsupported model output type for attention extraction.')

        attn = attn.detach()
        time_axis = attn.size(-1)
        output_axis = attn.size(1)

        text_lens_list = text_lens.tolist()
        mel_lens_list = reduced_mel_lens.tolist()

        for b in range(attn.size(0)):
            To = min(int(text_lens_list[b]), output_axis)
            Te = min(int(mel_lens_list[b]), time_axis)
            if To <= 1 or Te <= 1:
                continue

            a = attn[b, :To, :Te]
            total_mass = a.sum()
            if torch.isclose(total_mass, torch.tensor(0.0, device=a.device)):
                continue

            t = torch.arange(To, device=a.device, dtype=torch.float32).unsqueeze(1)
            e = torch.arange(Te, device=a.device, dtype=torch.float32).unsqueeze(0)
            t = t / (To - 1) if To > 1 else torch.zeros_like(t)
            e = e / (Te - 1) if Te > 1 else torch.zeros_like(e)
            diag = t - e
            mask = (diag.abs() <= band).to(a.dtype)

            score = (a * mask).sum() / total_mass.clamp_min(1e-8)
            diag_scores.append(float(score))

    mean_score = float(np.mean(diag_scores)) if diag_scores else float('nan')
    return mean_score, diag_scores


### Diagonal Attention Score
- scores > 0.6–0.7 are usually good; trending higher across training is a healthy sign.

In [ ]:
# evaluate the diagonal attention score
mean_score, scores = diagonal_attention_score(model, dev_loader, device=device, band=0.1, max_batches=None)
print(f'Diagonal attention score: {mean_score:.4f}')
print(f'Evaluated {len(scores)} alignments')

if scores:
    scores_array = np.array(scores)
    print('Score statistics:')
    print(f'  min:  {scores_array.min():.4f}')
    print(f'  25%:  {np.percentile(scores_array, 25):.4f}')
    print(f'  median: {np.median(scores_array):.4f}')
    print(f'  75%:  {np.percentile(scores_array, 75):.4f}')
    print(f'  max:  {scores_array.max():.4f}')
